# Model 2 — Pilot with REAL inspection outcomes (UK DVSA MOT open data)  (experiments only)

**Why.** Notebook 02 showed that Dataset 1 (and 4, 5) cannot support failure probabilities: their labels are rules. The SRS asks for ML on *real* failure information.
This pilot checks, with **real outcomes**, whether the same runtime features (vehicle age, odometer, make/model, fuel) predict **defects found at the annual roadworthiness test, by vehicle system**.

**What the outcome means (and does not).** An MOT defect is a failure *found at an annual inspection*, not a breakdown. The MOT groups defects by system:
brakes · lamps/reflectors/electrical equipment · noise, emissions and leaks · steering · suspension · tyres · etc.
So "engine" here can only mean *exhaust/emissions/leaks* and "electrical" only *lamps and electrical equipment*; **neither is engine health or battery health**.

**Data.** [DVSA anonymised MOT tests and results](https://open.data.dvsa.gov.uk/mot-anonymised/index.html), Open Government Licence v3.0, year **2023**.
A pseudo-random 1-in-41 sample of car tests (class 4, normal test) chosen by `test_id` was streamed from the remote files (see `ml/src/mot_pilot_sample.py`); the 1.2 GB / 0.46 GB files were not stored.

**Known limitation (found and handled).** The decoder silently corrupted the defect-items stream after the tests of 27 June 2023 (lines merge, values are garbage, no error is raised). Using those tests would have labelled defective vehicles as defect-free, so **only tests from 1 January to 26 June 2023 are used** (about 390 k car tests). Results do not cover the second half of the year; seasonal effects (for example winter lighting or tyre defects) are therefore not represented.
The UK also requires the first test at 3 years, like Israel (CBS Publication 1923, footnote 1 of Table 8), so the age profile of tested cars is comparable in structure.

**Not claimed:** that a UK-trained model is valid for Israel (different roads, climate, regulations, test standards). Israeli test outcomes are not open data; this pilot only tests whether the *approach* works on real outcomes.

In [1]:
import os, sys, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.options.display.width = 170
pd.options.display.max_columns = 40
pd.options.display.float_format = "{:,.4f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import INTERIM_DIR, RAW_DIR
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier, XGBRegressor
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
try:
    display
except NameError:
    display = print
YEAR = 2023

## 1. Data and system mapping

In [2]:
# The remote stream lost a few bytes at block boundaries (7,926 of 87 M item lines were unreadable). Damaged lines are dropped, and every kept row is validated:
# test_id must be an integer divisible by 41 (the sampling rule), otherwise the line is a damaged remnant.
res = pd.read_csv(INTERIM_DIR / f"uk_mot_{YEAR}_results_sample.csv", sep="|", low_memory=False, on_bad_lines="skip", dtype=str)
res = res[res.test_id.str.fullmatch(r"\d+")].copy(); res["test_id"] = res.test_id.astype("int64"); res = res[res.test_id % 41 == 0]
for c in ["vehicle_id", "test_mileage", "cylinder_capacity"]:
    res[c] = pd.to_numeric(res[c], errors="coerce")
res = res[(res.test_class_id == "4") & (res.test_type == "NT") & res.vehicle_id.notna()].drop_duplicates("test_id")
# DECODING LIMIT: the items stream is readable only up to the test of 2023-06-27 (about line 801,512 of the sample); beyond that the Deflate64 stream was silently corrupted
# (lines merge and values are garbage). Tests after that date would be given wrong "no defect" labels, so the analysis uses tests of 1 Jan - 26 Jun 2023 only.
CLEAN_ITEM_ROWS = 801_511
res = res[res.test_date < "2023-06-27"]
items = pd.read_csv(INTERIM_DIR / f"uk_mot_{YEAR}_items_sample.csv", sep="|", low_memory=False, on_bad_lines="skip", dtype=str, nrows=CLEAN_ITEM_ROWS)
items = items[items.test_id.str.fullmatch(r"\d+") & items.rfr_id.str.fullmatch(r"\d+")].copy()
items["test_id"] = items.test_id.astype("int64"); items["rfr_id"] = items.rfr_id.astype("int64"); items = items[items.test_id.isin(set(res.test_id))]
LK = RAW_DIR / "uk_mot_lookup"
grp = pd.read_csv(LK / "dft_item_group_extract.csv"); det = pd.read_csv(LK / "dft_item_detail_extract.csv")
print(f"sample: {len(res):,} car tests (NT), {items.test_id.nunique():,} tests with at least one recorded item, {len(items):,} item rows")
print("result codes:", res.test_result.value_counts(normalize=True).round(3).to_dict())
print("item type codes:", items.rfr_type_code.value_counts().to_dict(), "(F=fail, P=pass after rectification at station, A=advisory, M=minor)")
sec_names = grp[(grp.test_class_id == 4) & (grp.parent_id == 0)].set_index("test_item_set_section_id").item_name.to_dict()
rfr2sec = det[det.test_class_id == 4].drop_duplicates("rfr_id").set_index("rfr_id").test_item_set_section_id
items["section"] = items.rfr_id.map(rfr2sec)
SYSTEM = {5430: "brakes", 20003: "brakes", 5000: "electrical", 20158: "electrical", 5730: "engine_emissions", 20385: "engine_emissions"}
items["system"] = items.section.map(SYSTEM).fillna("other")
items["is_defect"] = items.rfr_type_code.isin(["F", "P"])
print("unmapped rfr ids:", int(items.section.isna().sum()))
sec_tbl = items[items.is_defect].assign(section_name=lambda d: d.section.map(sec_names)).groupby(["system", "section_name"]).test_id.nunique().rename("tests with a defect").reset_index().sort_values("tests with a defect", ascending=False)
display(sec_tbl.head(14))

sample: 376,346 car tests (NT), 224,525 tests with at least one recorded item, 800,129 item rows
result codes: {'P': 0.713, 'F': 0.223, 'PRS': 0.058, 'ABR': 0.005, 'ABA': 0.001, 'ABRVE': 0.0}
item type codes: {'A': 486562, 'F': 224975, 'M': 55275, 'P': 33317} (F=fail, P=pass after rectification at station, A=advisory, M=minor)


unmapped rfr ids: 0


,system,section_name,tests with a defect
1,electrical,"Lamps, reflectors and electrical equipment",41827
11,other,Suspension,32074
0,brakes,Brakes,25225
12,other,Tyres,23460
13,other,Visibility,19075
3,other,"Body, chassis, structure",12579
2,engine_emissions,"Noise, emissions and leaks",11427
10,other,Steering,6355
8,other,Seat belts and supplementary restraint systems,3985
6,other,Road Wheels,1334


In [3]:
d = res.copy()
d["test_date"] = pd.to_datetime(d.test_date, errors="coerce"); d["first_use_date"] = pd.to_datetime(d.first_use_date, errors="coerce")
d["age"] = (d.test_date - d.first_use_date).dt.days / 365.25
raw_n = len(d)
d = d[d.test_mileage.between(1, 600_000) & d.age.between(0, 30)]
FUEL = {"PE": "petrol", "DI": "diesel", "HY": "hybrid", "EL": "electric", "LP": "lpg_cng", "CN": "lpg_cng", "GB": "lpg_cng", "GA": "lpg_cng"}
d["fuel"] = d.fuel_type.map(FUEL).fillna("other")
d = d[d.fuel != "other"]
d["make"] = d.make.str.title(); d["model"] = d.make + " " + d.model.astype(str).str.title()
top_make = d.make.value_counts().head(60).index
d["make"] = d.make.where(d.make.isin(top_make), "OTHER")
top_model = d.model.value_counts().head(400).index
d["model"] = d.model.where(d.model.isin(top_model), "OTHER")
d["cc"] = pd.to_numeric(d.cylinder_capacity, errors="coerce")
print(f"rows: raw {raw_n:,} -> {len(d):,} after cleaning (mileage 1-600,000; age 0-30; known fuel)")
# outcomes per test
defects = items[items.is_defect]
for s in ["brakes", "electrical", "engine_emissions"]:
    d[s] = d.test_id.isin(set(defects[defects.system == s].test_id)).astype(int)
d["any_defect"] = d.test_id.isin(set(defects.test_id)).astype(int)
d["failed_or_prs"] = d.test_result.isin(["F", "PRS"]).astype(int)
TARGETS = {"brakes": "brakes", "electrical (lamps/electrical equipment)": "electrical", "engine-related (emissions/leaks)": "engine_emissions", "any defect": "any_defect"}
print("prevalence:", {k: round(float(d[v].mean()), 4) for k, v in TARGETS.items()}, "| test result F or PRS:", round(float(d.failed_or_prs.mean()), 4))
print("age (years):", d.age.describe()[["min", "25%", "50%", "75%", "max"]].round(1).to_dict(), "| mileage km?? NOTE: UK test_mileage is in MILES:", d.test_mileage.median())
d["km"] = d.test_mileage * 1.609344
print("median odometer (km):", round(float(d.km.median())))

rows: raw 376,346 -> 370,633 after cleaning (mileage 1-600,000; age 0-30; known fuel)


prevalence: {'brakes': 0.067, 'electrical (lamps/electrical equipment)': 0.1113, 'engine-related (emissions/leaks)': 0.0303, 'any defect': 0.2821} | test result F or PRS: 0.282
age (years): {'min': 0.0, '25%': 5.8, '50%': 8.9, '75%': 13.1, 'max': 30.0} | mileage km?? NOTE: UK test_mileage is in MILES: 63127.0
median odometer (km): 101593


## 2. Protocol
* Split **by vehicle** (`vehicle_id`) 70 / 15 / 15 into train / validation / test, so one car never sits in two parts.
* **Scenarios (features):**
  * **G** — registry-like: age, fuel, make (what exists for an arbitrary plate).
  * **G+model** — plus the model name (the registry has `kinuy_mishari`).
  * **E (Stage-1 estimate)** — G plus an *estimated* odometer from a regression trained on the training part with the same registry-like features (this reproduces the SRS two-stage pipeline: Stage 1 feeds Stage 2).
  * **M (true odometer)** — G plus the real odometer reading (the ceiling when Stage 1 is perfect).
* **Models:** XGBoost on the natural (un-weighted) objective so probabilities stay interpretable; Logistic Regression baseline (SRS); and, for brakes, class-weight vs SMOTE (PDF requirement).
* **Metrics:** ROC-AUC, PR-AUC (prevalence in brackets), top-decile lift, **capture** (share of true defects found among the 20% highest-scored vehicles), calibration error, Brier. No threshold is optimised on the test part.

In [4]:
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=0)
tr_idx, tmp_idx = next(gss.split(d, groups=d.vehicle_id))
tmp = d.iloc[tmp_idx]
va_rel, te_rel = next(GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=0).split(tmp, groups=tmp.vehicle_id))
tr, va, te = d.iloc[tr_idx].copy(), tmp.iloc[va_rel].copy(), tmp.iloc[te_rel].copy()
print(f"train {len(tr):,} | validation {len(va):,} | test {len(te):,}")

# Stage-1 style estimate of the odometer (km) from registry-like features, trained on the training part only
def to_cat(df):
    df = df.copy()
    for c in ["fuel", "make", "model"]:
        df[c] = pd.Categorical(df[c], categories=sorted(d[c].unique()))
    return df
tr, va, te = to_cat(tr), to_cat(va), to_cat(te)
reg = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.8, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0)
reg.fit(tr[["age", "fuel", "make"]], tr.km)
for df in (tr, va, te):
    df["km_est"] = reg.predict(df[["age", "fuel", "make"]])
from sklearn.metrics import r2_score
print(f"Stage-1 style odometer estimate on the test part: R2 = {r2_score(te.km, te.km_est):.3f}, RMSE = {np.sqrt(np.mean((te.km - te.km_est)**2)):,.0f} km")
SCEN = {"G": ["age", "fuel", "make"], "G+model": ["age", "fuel", "make", "model"], "E (Stage-1 estimate)": ["age", "fuel", "make", "model", "km_est"], "M (true odometer)": ["age", "fuel", "make", "model", "km"]}

train 259,435 | validation 55,600 | test 55,598


Stage-1 style odometer estimate on the test part: R2 = 0.508, RMSE = 51,484 km


In [5]:
def ece(y, p, bins=10):
    q = pd.qcut(p, bins, duplicates="drop")
    t = pd.DataFrame({"y": y, "p": p}).groupby(q, observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return float((abs(t.p - t.y) * t.n).sum() / t.n.sum()), t

def capture(y, p, frac=0.2):
    k = int(len(p) * frac); idx = np.argsort(-p)[:k]
    return float(np.asarray(y)[idx].sum() / np.asarray(y).sum())

def fit_xgb(Xtr, ytr, **kw):
    m = XGBClassifier(n_estimators=250, max_depth=6, learning_rate=0.06, subsample=0.8, colsample_bytree=0.8, min_child_weight=20,
                      tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0, **kw)
    return m.fit(Xtr, ytr)

rows, store = [], {}
for tname, tcol in TARGETS.items():
    for sname, cols in SCEN.items():
        m = fit_xgb(tr[cols], tr[tcol]); p = m.predict_proba(te[cols])[:, 1]
        e, _ = ece(te[tcol], p)
        rows.append({"system": tname, "features": sname, "prevalence": te[tcol].mean(), "ROC-AUC": roc_auc_score(te[tcol], p), "PR-AUC": average_precision_score(te[tcol], p),
                     "top-decile lift": float(te[tcol].to_numpy()[np.argsort(-p)[:len(p)//10]].mean() / te[tcol].mean()),
                     "capture in top 20%": capture(te[tcol], p), "calibration err": e, "Brier": brier_score_loss(te[tcol], p)})
        store[(tname, sname)] = (m, p)
results = pd.DataFrame(rows)
display(results)

,system,features,prevalence,ROC-AUC,PR-AUC,top-decile lift,capture in top 20%,calibration err,Brier
0,brakes,G,0.0677,0.6897,0.1303,2.2491,0.4022,0.0032,0.0612
1,brakes,G+model,0.0677,0.6951,0.1386,2.4379,0.4117,0.0037,0.0610
2,brakes,E (Stage-1 estimate),0.0677,0.6936,0.1369,2.4565,0.4062,0.0049,0.0611
3,brakes,M (true odometer),0.0677,0.7115,0.1456,2.6187,0.4256,0.0046,0.0607
4,electrical (lamps/electrical equipment),G,0.1099,0.7164,0.2112,2.2324,0.4098,0.0040,0.0923
5,electrical (lamps/electrical equipment),G+model,0.1099,0.7261,0.2201,2.3093,0.4206,0.0050,0.0917
6,electrical (lamps/electrical equipment),E (Stage-1 estimate),0.1099,0.7258,0.2189,2.2995,0.4228,0.0046,0.0917
7,electrical (lamps/electrical equipment),M (true odometer),0.1099,0.7427,0.2357,2.5548,0.4508,0.0047,0.0906
8,engine-related (emissions/leaks),G,0.0300,0.7308,0.0779,2.8944,0.4847,0.0013,0.0284
9,engine-related (emissions/leaks),G+model,0.0300,0.7394,0.0790,2.8584,0.4937,0.0025,0.0284


## 3. Baselines required by the PDFs: Logistic Regression vs XGBoost; class weights vs SMOTE (brakes, true-odometer scenario)

In [6]:
cols = SCEN["M (true odometer)"]
def lr_pipe():
    num = ["age", "km"]; cat = ["fuel", "make"]
    return Pipeline([("p", ColumnTransformer([("n", StandardScaler(), num), ("c", OneHotEncoder(handle_unknown="ignore", min_frequency=200, sparse_output=False), cat)])),
                     ("m", LogisticRegression(max_iter=1000))])
tcol = "brakes"; prev = te[tcol].mean(); rows = []
lr = lr_pipe().fit(tr[["age", "km", "fuel", "make"]].astype({"fuel": str, "make": str}), tr[tcol])
p_lr = lr.predict_proba(te[["age", "km", "fuel", "make"]].astype({"fuel": str, "make": str}))[:, 1]
rows.append({"model": "Logistic Regression (baseline)", "ROC-AUC": roc_auc_score(te[tcol], p_lr), "PR-AUC": average_precision_score(te[tcol], p_lr), "capture in top 20%": capture(te[tcol], p_lr), "calibration err": ece(te[tcol], p_lr)[0]})
p_x = store[("brakes", "M (true odometer)")][1]
rows.append({"model": "XGBoost, natural objective", "ROC-AUC": roc_auc_score(te[tcol], p_x), "PR-AUC": average_precision_score(te[tcol], p_x), "capture in top 20%": capture(te[tcol], p_x), "calibration err": ece(te[tcol], p_x)[0]})
pw = (tr[tcol] == 0).sum() / (tr[tcol] == 1).sum()
mw = fit_xgb(tr[cols], tr[tcol], scale_pos_weight=pw); p_w = mw.predict_proba(te[cols])[:, 1]
rows.append({"model": f"XGBoost, scale_pos_weight={pw:.1f}", "ROC-AUC": roc_auc_score(te[tcol], p_w), "PR-AUC": average_precision_score(te[tcol], p_w), "capture in top 20%": capture(te[tcol], p_w), "calibration err": ece(te[tcol], p_w)[0]})
# SMOTE on a 150k subsample of the training part (numeric + one-hot make/fuel)
sub = tr.sample(150_000, random_state=0)
Xs = pd.get_dummies(sub[["age", "km", "fuel", "make"]].astype({"fuel": str, "make": str}), dtype=float)
cols_s = Xs.columns
sm = ImbPipeline([("s", SMOTE(random_state=0)), ("m", XGBClassifier(n_estimators=250, max_depth=6, learning_rate=0.06, subsample=0.8, tree_method="hist", n_jobs=8, random_state=0))]).fit(Xs, sub[tcol])
Xt = pd.get_dummies(te[["age", "km", "fuel", "make"]].astype({"fuel": str, "make": str}), dtype=float).reindex(columns=cols_s, fill_value=0.0)
p_s = sm.predict_proba(Xt)[:, 1]
rows.append({"model": "XGBoost + SMOTE (150k subsample)", "ROC-AUC": roc_auc_score(te[tcol], p_s), "PR-AUC": average_precision_score(te[tcol], p_s), "capture in top 20%": capture(te[tcol], p_s), "calibration err": ece(te[tcol], p_s)[0]})
display(pd.DataFrame(rows).assign(**{"prevalence": prev}))

,model,ROC-AUC,PR-AUC,capture in top 20%,calibration err,prevalence
0,Logistic Regression (baseline),0.6992,0.1339,0.4187,0.0061,0.0677
1,"XGBoost, natural objective",0.7115,0.1456,0.4256,0.0046,0.0677
2,"XGBoost, scale_pos_weight=13.9",0.7095,0.1471,0.4250,0.3433,0.0677
3,XGBoost + SMOTE (150k subsample),0.6950,0.1266,0.3886,0.0415,0.0677


## 4. Are the probabilities usable? Reliability of the natural-objective XGBoost (brakes, estimated-odometer scenario = what the SRS pipeline would feed)

In [7]:
m, p = store[("brakes", "E (Stage-1 estimate)")]
e, tbl = ece(te["brakes"], p)
print(f"expected calibration error = {e:.4f}")
display(tbl.rename(columns={"p": "mean predicted", "y": "observed rate"}))
imp = pd.Series(m.feature_importances_, index=SCEN["E (Stage-1 estimate)"]).sort_values(ascending=False)
print("feature importance (gain share):", imp.round(3).to_dict())

expected calibration error = 0.0049


,mean predicted,observed rate,n
"(-0.000794, 0.0143]",0.0086,0.0142,5564
"(0.0143, 0.0238]",0.0193,0.0239,5556
"(0.0238, 0.0327]",0.0282,0.0325,5563
"(0.0327, 0.0438]",0.0380,0.0410,5556
"(0.0438, 0.0555]",0.0495,0.0516,5560
"(0.0555, 0.0685]",0.0619,0.0603,5560
"(0.0685, 0.0829]",0.0755,0.0854,5559
"(0.0829, 0.105]",0.0933,0.0926,5560
"(0.105, 0.136]",0.1196,0.1089,5563
"(0.136, 0.477]",0.1722,0.1661,5557


feature importance (gain share): {'age': 0.32899999618530273, 'model': 0.30300000309944153, 'make': 0.1459999978542328, 'km_est': 0.14100000262260437, 'fuel': 0.07999999821186066}


## 4b. Precision, Recall and F1 with a threshold chosen on the validation part (never on test), against the trivial "flag every vehicle" baseline, for the pipeline the SRS describes (Stage-1 estimate -> Stage 2)
The threshold maximises F1 on validation. Logistic Regression is trained on the same features.

In [8]:
from sklearn.metrics import precision_recall_curve, f1_score
def best_f1_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p); f1 = 2 * pr * rc / np.clip(pr + rc, 1e-12, None); i = int(np.nanargmax(f1[:-1])); return float(th[i])
def lr_fit(df, tcol):
    pipe = Pipeline([("p", ColumnTransformer([("n", StandardScaler(), ["age", "km_est"]), ("c", OneHotEncoder(handle_unknown="ignore", min_frequency=200, sparse_output=False), ["fuel", "make"])])), ("m", LogisticRegression(max_iter=1000))])
    return pipe.fit(df[["age", "km_est", "fuel", "make"]].astype({"fuel": str, "make": str}), df[tcol])
rows = []
cols = SCEN["E (Stage-1 estimate)"]
for tname, tcol in TARGETS.items():
    mx = fit_xgb(tr[cols], tr[tcol]); px_va, px_te = mx.predict_proba(va[cols])[:, 1], mx.predict_proba(te[cols])[:, 1]
    lr = lr_fit(tr, tcol)
    X = lambda df: df[["age", "km_est", "fuel", "make"]].astype({"fuel": str, "make": str})
    pl_va, pl_te = lr.predict_proba(X(va))[:, 1], lr.predict_proba(X(te))[:, 1]
    y = te[tcol].to_numpy(); prev = y.mean()
    for nm, pva, pte in [("XGBoost", px_va, px_te), ("Logistic Regression", pl_va, pl_te)]:
        th = best_f1_threshold(va[tcol], pva); pred = (pte >= th).astype(int); tp = (pred & y).sum()
        rows.append({"system": tname, "model": nm, "ROC-AUC": roc_auc_score(y, pte), "threshold": th, "share flagged": pred.mean(), "Precision": tp / max(pred.sum(), 1), "Recall": tp / y.sum(), "F1": f1_score(y, pred)})
    rows.append({"system": tname, "model": "baseline: flag every vehicle", "ROC-AUC": 0.5, "threshold": 0.0, "share flagged": 1.0, "Precision": prev, "Recall": 1.0, "F1": 2 * prev / (1 + prev)})
display(pd.DataFrame(rows))

,system,model,ROC-AUC,threshold,share flagged,Precision,Recall,F1
0,brakes,XGBoost,0.6936,0.0954,0.2419,0.1302,0.4654,0.2035
1,brakes,Logistic Regression,0.6868,0.0970,0.2234,0.1327,0.4381,0.2037
2,brakes,baseline: flag every vehicle,0.5000,0.0000,1.0000,0.0677,1.0000,0.1268
3,electrical (lamps/electrical equipment),XGBoost,0.7258,0.1481,0.3188,0.2091,0.6066,0.3110
4,electrical (lamps/electrical equipment),Logistic Regression,0.7097,0.1469,0.2817,0.2068,0.5300,0.2975
5,electrical (lamps/electrical equipment),baseline: flag every vehicle,0.5000,0.0000,1.0000,0.1099,1.0000,0.1981
6,engine-related (emissions/leaks),XGBoost,0.7397,0.0862,0.0617,0.1035,0.2127,0.1392
7,engine-related (emissions/leaks),Logistic Regression,0.7114,0.0563,0.1398,0.0754,0.3511,0.1242
8,engine-related (emissions/leaks),baseline: flag every vehicle,0.5000,0.0000,1.0000,0.0300,1.0000,0.0583
9,any defect,XGBoost,0.6753,0.2408,0.5613,0.3730,0.7482,0.4978


## 4c. Stability of the result: re-split with 3 different random seeds (grouped by vehicle), AUC for the pipeline with the Stage-1 estimate

In [9]:
rows = []
for seed in [1, 2, 3]:
    a, b = next(GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=seed).split(d, groups=d.vehicle_id))
    trs, tes = to_cat(d.iloc[a].copy()), to_cat(d.iloc[b].copy())
    r = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.8, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0).fit(trs[["age", "fuel", "make"]], trs.km)
    trs["km_est"], tes["km_est"] = r.predict(trs[["age", "fuel", "make"]]), r.predict(tes[["age", "fuel", "make"]])
    for tname, tcol in TARGETS.items():
        m = fit_xgb(trs[SCEN["E (Stage-1 estimate)"]], trs[tcol]); rows.append({"seed": seed, "system": tname, "ROC-AUC": roc_auc_score(tes[tcol], m.predict_proba(tes[SCEN["E (Stage-1 estimate)"]])[:, 1])})
sd = pd.DataFrame(rows).pivot(index="system", columns="seed", values="ROC-AUC"); sd["mean"] = sd.mean(axis=1); sd["spread (max-min)"] = sd[[1, 2, 3]].max(axis=1) - sd[[1, 2, 3]].min(axis=1)
display(sd)

seed,1,2,3,mean,spread (max-min)
system,,,,,
any defect,0.6770,0.6774,0.6750,0.6765,0.0024
brakes,0.6921,0.6904,0.6939,0.6922,0.0035
electrical (lamps/electrical equipment),0.7231,0.7243,0.7254,0.7243,0.0023
engine-related (emissions/leaks),0.7318,0.7359,0.7343,0.7340,0.0040


## 4d. Is Stage 1 useful at all? What the estimated odometer carries beyond age and fuel
In Israel the Stage-1 value is a function of age, fuel (and, after calibration, ownership). The comparison G+model vs E in the table above (identical AUC to 3 decimals) is the evidence; the true odometer (M) is what would add information.

## 5. How much is lost by using an estimated odometer instead of the true one?

In [10]:
pv = results.pivot_table(index="system", columns="features", values="ROC-AUC")[list(SCEN)]
display(pv)
pc = results.pivot_table(index="system", columns="features", values="capture in top 20%")[list(SCEN)]
display(pc)

features,G,G+model,E (Stage-1 estimate),M (true odometer)
system,,,,
any defect,0.6721,0.6766,0.6753,0.6910
brakes,0.6897,0.6951,0.6936,0.7115
electrical (lamps/electrical equipment),0.7164,0.7261,0.7258,0.7427
engine-related (emissions/leaks),0.7308,0.7394,0.7397,0.7549


features,G,G+model,E (Stage-1 estimate),M (true odometer)
system,,,,
any defect,0.3227,0.3286,0.3279,0.3421
brakes,0.4022,0.4117,0.4062,0.4256
electrical (lamps/electrical equipment),0.4098,0.4206,0.4228,0.4508
engine-related (emissions/leaks),0.4847,0.4937,0.4985,0.5219
